<div align="center">

# Week 01: Foundations: The Interaction Loop, User-Centred Design and Usability

**Human-Computer Interaction (BIL-429), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-01) · [Lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/lecture.html) · [Interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/lab.html) · [PDF notes](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/Week01_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also defines `CLINIC_FLOW`, the booking flow of the clinic app with the values of the lecture. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok


# The booking flow of the clinic app in Week 1, with illustrative values per step:
#   t   time of the step when it goes well, in seconds
#   e   probability of a slip at the step
#   r   extra time to notice and repair a slip, in seconds
#   a   probability of giving up after a slip
_FLOW_ROWS = [
    # step                                             t     e    r     a
    ("Log in with the patient number and a password", 20, 0.10, 30, 0.20),
    ("Find Appointments under the Services menu",       8, 0.25, 15, 0.10),
    ("Choose the department from a list of 40",        12, 0.20, 10, 0.10),
    ("Choose the doctor",                               6, 0.05,  6, 0.05),
    ("Pick a date in the calendar",                    10, 0.15, 12, 0.10),
    ("Pick a time slot",                                5, 0.08,  6, 0.05),
    ("Type the patient number again",                  15, 0.12, 20, 0.15),
    ("Press Book, with no confirmation screen",         4, 0.10, 25, 0.10),
]
CLINIC_FLOW = [dict(step=s, t=t, e=e, r=r, a=a) for s, t, e, r, a in _FLOW_ROWS]
print("Setup complete. Running in Colab:", IN_COLAB)

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 1.1": (lambda: 58.0, 1e-6),
    "Exercise 1.2": (lambda: 0.6, 1e-6),
    "Exercise 1.3": (lambda: (140 + 95 + 210) / 3, 1e-6),
    "Exercise A": (lambda: 85.0, 1e-6),
    "Exercise B": (lambda: float((kiosk["errors"] > 0).mean()), 1e-6),
    "Exercise C": (lambda: float(door_sum.loc["handle_push", "error_rate"]), 1e-3),
    "Exercise D": (lambda: 0.9 * 0.925 * 0.94, 1e-6),
    "Exercise E": (lambda: int(table["e*a (gives up)"].idxmax()), 1e-6),
}

## Python step 1: Values, variables, lists and decisions

A Colab notebook is a sequence of cells. Text cells explain, and code cells hold Python instructions. A code cell runs when it is executed with Shift+Enter, and its output appears directly below it &#91;[4](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/lecture.html#ref-4)&#93;. This step works with data that the kiosk of the clinic app records: How long a booking took, how many taps it needed and who used it. It computes with single values first, then with lists of many sessions, and ends by comparing two groups of patients &#91;[5](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/lecture.html#ref-5)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "CLINIC_FLOW" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-01/"
               "NB01_interaction_loop_and_usability.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Values and variables

In [ ]:
# Python practice: a small example for learning the language
taps = 14                  # taps on the kiosk screen: an integer
seconds = 95.5             # time from the first to the last tap: a float
patient = "Ayşe Yılmaz"    # text: a string
booked = True              # did the booking work? a boolean
taps_per_minute = taps / seconds * 60
print(patient, "| booked:", booked, "|", type(taps), type(seconds))
print(f"{taps} taps in {seconds} s = {taps_per_minute:.1f} taps per minute")
print(f"one tap every {seconds / taps:.1f} s on average")

A variable is a name bound to a value with `=`. A whole number is an integer (`int`), and a number with a decimal point is a `float`. Text in quotation marks is a string (`str`), and `True` or `False` is a boolean (`bool`). Arithmetic uses `+`, `-`, `*` and `/`, and `type` tells the kind of a value. In the f-string `f"..."`, an expression in braces is computed and inserted into the text, and `:.1f` shows it with one decimal. The text after `#` is a comment, which Python ignores.

### A decision is an if statement

In [ ]:
# Python practice: a small example for learning the language
seconds = 95.5
target_s = 60              # the hospital wants a booking within one minute
if seconds <= target_s:
    verdict = "within the target"
else:
    over = seconds - target_s
    verdict = f"{over:.1f} s over the target, {over / target_s:.0%} too slow"
print(f"A booking of {seconds} s is {verdict}.")

An `if` statement runs the indented block below it only when its condition is true, and `else` covers the other case. The comparison `<=` means less than or equal to. Here the decision also computes how far a booking misses the target, in seconds and as a share of the target, which `:.0%` prints as a percentage.

### Lists and loops

In [ ]:
# Python practice: a small example for learning the language
times_s = [48, 95, 62, 140, 55, 71, 38, 210, 66, 59]   # ten patients at the kiosk
total = 0
slow = 0
for t in times_s:
    total = total + t
    if t > 120:
        slow = slow + 1
mean_s = total / len(times_s)
print(f"mean {mean_s:.1f} s | slowest {max(times_s)} s | fastest {min(times_s)} s")
print(f"{slow} of {len(times_s)} patients needed more than two minutes")

A list holds values in order between square brackets. The `for` loop takes one value after another and runs its indented block for each, so one loop adds up all times and counts the slow sessions at once. `len` counts the elements, and `max` and `min` find the largest and the smallest. Ten values could be added by hand, but a kiosk records hundreds of sessions a day, and the same three lines work for all of them.

### Functions

In [ ]:
# Python practice: a small example for learning the language
def share_within(times, limit):
    """Share of the times that are at most limit, between 0 and 1."""
    if len(times) == 0:
        return 0.0
    count = 0
    for t in times:
        if t <= limit:
            count = count + 1
    return count / len(times)

for limit in [60, 90, 120]:
    print(f"{share_within(times_s, limit):.0%} of the patients booked within {limit} s")

`def` defines a function with parameters in brackets and a `return` statement that hands back the result. The text in triple quotes is a docstring, which describes the function. The first `if` protects against a division by zero for an empty list. Once defined, the function answers the same question for any list and any limit, as the loop over three limits shows.

### Dictionaries

In [ ]:
# Python practice: a small example for learning the language
kiosk_times = {"old kiosk": [95, 140, 210, 120], "new kiosk": [48, 62, 55, 59]}
kiosk_times["new kiosk"].append(51)          # one more patient on the new kiosk
means = {}
for design, times in kiosk_times.items():
    means[design] = sum(times) / len(times)
    print(f"{design}: {len(times)} patients, mean {means[design]:.1f} s")
saving = means["old kiosk"] - means["new kiosk"]
print(f"the new kiosk saves {saving:.1f} s per booking")

A dictionary maps keys to values between curly braces. `kiosk_times["new kiosk"]` looks up the value of a key, here a list, and `append` adds an element to the end of that list. `items()` gives the pairs of keys and values to a loop. Assigning to a new key, as in `means[design] = ...`, adds the key. The result is the difference between two designs, the number that the lecture turns into a yearly cost.

### A list of dictionaries

In [ ]:
# Python practice: a small example for learning the language
sessions = [
    {"patient": "P01", "age": 31, "time_s": 48},
    {"patient": "P02", "age": 74, "time_s": 140},
    {"patient": "P03", "age": 45, "time_s": 62},
    {"patient": "P04", "age": 68, "time_s": 95},
    {"patient": "P05", "age": 23, "time_s": 38},
    {"patient": "P06", "age": 81, "time_s": 210},
]
older, younger = [], []
for s in sessions:
    if s["age"] >= 65:
        older.append(s["time_s"])
    else:
        younger.append(s["time_s"])
mean_older = sum(older) / len(older)
mean_younger = sum(younger) / len(younger)
print(f"65 or older: {len(older)} patients, mean {mean_older:.1f} s")
print(f"under 65:    {len(younger)} patients, mean {mean_younger:.1f} s")
print(f"the older group needs {mean_older / mean_younger:.1f} times as long")

A list of dictionaries keeps one record per session, with named fields. The loop sorts each session into one of two lists by a decision on its age, and the means of the two lists compare the groups. The same kiosk costs the older group three times as much time. This is a fact about people rather than about the software, and the lecture takes it up in the section on the human side of the loop. The Colab notebook stores the booking flow of the clinic app in the same way, one dictionary per step.

**Quick check.** What does `share_within([], 60)` return, and why?

<details><summary>Answer</summary>

`0.0`. The list is empty, so the condition `len(times) == 0` is true, and the function returns 0.0 before it would divide by zero.

</details>

### Exercises for Python step 1

**Exercise 1.1.** Five rounds on a kiosk took 52, 75, 61, 180 and 44 seconds, but the round of 180 seconds ended without a booking. Store the mean time of the four completed rounds in `mean_ok`. The loop `for t, ok in zip(round_times, round_booked):` takes each time together with its outcome.

In [ ]:
round_times = [52, 75, 61, 180, 44]
round_booked = [True, True, True, False, True]
mean_ok = None   # your answer

check("Exercise 1.1", mean_ok)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
round_times = [52, 75, 61, 180, 44]
round_booked = [True, True, True, False, True]
total, count = 0, 0
for t, ok in zip(round_times, round_booked):
    if ok:
        total = total + t
        count = count + 1
mean_ok = total / count

check("Exercise 1.1", mean_ok)

**Exercise 1.2.** Write a function `share_over(times, limit)` that returns the share of the times above `limit`, and store `share_over(times_s, 60)` in `p_over`.

In [ ]:
def share_over(times, limit):
    # your code
    return None

p_over = share_over(times_s, 60)

check("Exercise 1.2", p_over)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def share_over(times, limit):
    if len(times) == 0:
        return 0.0
    count = 0
    for t in times:
        if t > limit:
            count = count + 1
    return count / len(times)

p_over = share_over(times_s, 60)

check("Exercise 1.2", p_over)

**Exercise 1.3.** Use the list `sessions` to store the mean time of the patients aged 65 or more in `mean_old`. Compute it with a loop and a decision instead of copying it from the output above.

In [ ]:
mean_old = None   # your code

check("Exercise 1.3", mean_old)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
total, count = 0, 0
for s in sessions:
    if s["age"] >= 65:
        total = total + s["time_s"]
        count = count + 1
mean_old = total / count

check("Exercise 1.3", mean_old)

## 1. The two kiosks: Usability in numbers &#91;[3](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/lecture.html#ref-3)&#93;

*Lecture: [Usability: Effectiveness, efficiency and satisfaction](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/lecture.html#usability-effectiveness-efficiency-and-satisfaction)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "sessions" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-01/"
               "NB01_interaction_loop_and_usability.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A usability test produces many numbers. Every round has a time, an outcome and a count of errors. Every questionnaire of the System Usability Scale (SUS) has ten answers. A class of 24 students who play two kiosks three times each produces 144 rounds and 480 answers. Python turns them into the four measures of the lecture in one run, and a new test needs nothing but new data.

**From the formulas to the code.** In the code, $n$ is `n`, $s$ is `s`, $E$ is `E`, $\bar{t}_s$ is `t_s` and $\eta$ is `eta`, in tasks per minute. The set $S$ of the successful participants becomes the selection `runs["completed"] == 1`. In `sus_score`, the contribution $c_j$ is `x - 1` for an odd statement and `5 - x` for an even one, and the factor 2.5 multiplies their sum.

In [ ]:
def usability_measures(runs):
    """E, the mean time of the completed runs in s and the efficiency in tasks per
    minute, for a table with the columns completed (0 or 1) and time_s."""
    n = len(runs)
    s = int(runs["completed"].sum())
    t_s = runs.loc[runs["completed"] == 1, "time_s"].mean()
    eta = s / (runs["time_s"].sum() / 60)
    return {"n": n, "s": s, "E": s / n, "t_s": t_s, "eta": eta}

def sus_score(answers):
    """SUS score of one questionnaire: ten answers from 1 to 5, statement 1 first."""
    total = 0
    for j, x in enumerate(answers, start=1):
        if j % 2 == 1:           # odd statements are positive
            total += x - 1
        else:                    # even statements are negative
            total += 5 - x
    return 2.5 * total

# Both functions checked with the worked examples of the lecture
times_min = [2.1, 2.6, 3.0, 3.2, 3.5, 3.9, 4.4, 5.0, 5.6, 4.9, 5.5, 6.7]
example = pd.DataFrame({"completed": [1] * 9 + [0] * 3,
                        "time_s": [60 * t for t in times_min]})
m = usability_measures(example)
print(f"E = {m['E']:.2f} | mean time {m['t_s'] / 60:.2f} min | "
      f"eta = {m['eta']:.3f} tasks per minute")
print("SUS of the example patient:", sus_score([4, 2, 4, 1, 3, 2, 4, 2, 3, 3]))

**Python note.** `enumerate(answers, start=1)` numbers the answers from 1, so `j` is the number of the statement. `j % 2` is the remainder of the division by 2, which is 1 for odd numbers. `total += x - 1` is short for `total = total + (x - 1)`. A dictionary returns several results of a function at once, and `runs.loc[condition, "time_s"]` selects the times of the rows that meet the condition.

Part A of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text between the triple quotes of `MY_KIOSK` and run the cells. With `MY_KIOSK` empty, the code uses a simulated class of 24 students. Their times, errors and answers are assumptions of this Colab notebook, not measurements, and half of them play the system-centred kiosk first.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_KIOSK = """
"""

def simulated_kiosk_class(n_students=24, seed=RANDOM_STATE):
    """Three rounds per kiosk and student, with assumed times, errors and answers."""
    rng = np.random.default_rng(seed)
    # median time of a round in s, probability of failing, mean errors per round
    setup = {"system": (92.0, 0.22, 2.4), "user": (48.0, 0.04, 0.4)}
    # mean answer to the positive and to the negative statements
    answers = {"system": (2.6, 3.4), "user": (4.1, 1.8)}
    rows = []
    for student in range(1, n_students + 1):
        order = ["system", "user"] if student % 2 else ["user", "system"]
        speed = rng.lognormal(0, 0.2)                # some students are faster
        for pos, design in enumerate(order, start=1):
            t_med, p_fail, err = setup[design]
            learn = 0.9 if pos == 2 else 1.0         # the second kiosk feels familiar
            mu = [answers[design][j % 2] for j in range(10)]
            sus = np.clip(np.round(rng.normal(mu, 0.8)), 1, 5).astype(int)
            for rnd in range(1, 4):
                t = t_med * speed * learn * rng.lognormal(0, 0.25)
                rows.append([student, 1 if design == "system" else 2, design, pos,
                             rnd, int(rng.random() > p_fail), round(t, 1),
                             int(rng.poisson(err))] + list(sus))
    cols = (["student", "kiosk", "design", "order", "round", "completed", "time_s",
             "errors"] + [f"sus{j}" for j in range(1, 11)])
    return pd.DataFrame(rows, columns=cols)

if MY_KIOSK.strip():
    kiosk = pd.read_csv(io.StringIO(MY_KIOSK.strip()))
    kiosk["student"] = 1
    source = "your own rounds from the interactive lab"
else:
    kiosk = simulated_kiosk_class()
    source = "a simulated class of 24 students"
print("Data:", source, "|", len(kiosk), "rounds")
kiosk.head()

**Python note.** `"""..."""` is a string that may span several lines, and `io.StringIO` lets `pd.read_csv` read the pasted text like a file. `rng.lognormal` draws positive, right-skewed times, as real task times are, and `rng.poisson` draws whole numbers of errors. `np.clip(np.round(...), 1, 5)` turns random numbers into answers on the five-point scale.

In [ ]:
sus_cols = [f"sus{j}" for j in range(1, 11)]
forms = kiosk.groupby(["student", "design"])[sus_cols].first().dropna()
forms["SUS"] = [sus_score(a) for a in forms[sus_cols].astype(int).values]

summary = pd.DataFrame({d: usability_measures(r)
                        for d, r in kiosk.groupby("design")}).T.astype(float)
summary["errors_per_round"] = kiosk.groupby("design")["errors"].mean()
summary["SUS"] = forms.groupby("design")["SUS"].mean()
summary[["n", "s"]] = summary[["n", "s"]].astype(int)
print(summary.round(3))

if {"system", "user"} <= set(summary.index):
    dt = summary.loc["system", "t_s"] - summary.loc["user", "t_s"]
    N, D = 600, 250        # bookings per day and working days per year, as in the lecture
    C_hours = N * D * dt / 3600
    print(f"difference in the mean time of completed rounds: {dt:.1f} s")
    print(f"yearly cost C = {N} x {D} x {dt:.1f} s = {C_hours:,.0f} hours")
else:
    print("Play both kiosks in the interactive lab to compare them.")

**Python note.** `groupby(["student", "design"])` makes one group per student and kiosk, and `first()` keeps one questionnaire of each group, because every round of a kiosk repeats the same answers. The dictionary comprehension `{d: ... for d, r in kiosk.groupby("design")}` applies `usability_measures` to the rounds of each design, and `.T` turns the result so that each design becomes a row. `{"system", "user"} <= set(...)` asks whether both designs are present, and `:,.0f` prints a number with a thousands separator.

In [ ]:
# Figure: the two kiosks side by side
designs, names = ["system", "user"], ["system-centred", "user-centred"]
done = kiosk[kiosk["completed"] == 1]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.9))
times = [done.loc[done["design"] == d, "time_s"] for d in designs]
ax[0].boxplot(times, showfliers=False, widths=0.5)
for i, t in enumerate(times, start=1):
    x = i + np.random.default_rng(i).uniform(-0.15, 0.15, len(t))
    ax[0].scatter(x, t, s=12, alpha=0.5, color="#5A4BB5")
ax[0].set_xticks([1, 2], names)
ax[0].set_ylabel("time of a completed round (s)")
ax[0].set_title("Time on task")
wide = forms["SUS"].unstack("design").reindex(columns=designs)
for _, r in wide.dropna().iterrows():
    ax[1].plot([1, 2], [r["system"], r["user"]], color="#B9B4CC", lw=1)
ax[1].scatter(np.ones(len(wide)), wide["system"], color="#8A93A3", zorder=3)
ax[1].scatter(np.full(len(wide), 2), wide["user"], color="#5A4BB5", zorder=3)
ax[1].axhline(68, ls="--", color="#C98510", label="mean of published studies, 68")
ax[1].set_xticks([1, 2], names)
ax[1].set_ylim(0, 102)
ax[1].set_ylabel("SUS score")
ax[1].set_title("Satisfaction: the two scores of each student")
ax[1].legend(loc="lower right")
plt.tight_layout()
plt.show()

**Python note.** A box plot shows the middle half of the values as a box with the median inside, and the scattered dots show every completed round. `unstack("design")` turns the designs into columns, so that each row holds the two scores of one student. `iterrows()` gives the rows to a loop that joins each pair with a line.

**Exercise A.** A patient answered the ten statements for the user-centred kiosk with 5, 1, 4, 2, 4, 1, 5, 2, 4 and 2. Compute the SUS score by hand first, then with `sus_score`, and store it in `sus_patient`.

In [ ]:
sus_patient = None   # your answer

check("Exercise A", sus_patient)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
sus_patient = sus_score([5, 1, 4, 2, 4, 1, 5, 2, 4, 2])

check("Exercise A", sus_patient)

**Exercise B.** Store the share of all rounds, of both kiosks, that had at least one error in `p_any_error`. The comparison `kiosk["errors"] > 0` gives `True` or `False` for every round, and `.mean()` of such a column is the share of `True`.

In [ ]:
p_any_error = None   # your answer

check("Exercise B", p_any_error)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_any_error = (kiosk["errors"] > 0).mean()

check("Exercise B", p_any_error)

**Observe and reflect.** The page drew the order of the kiosks at random, and half of the simulated students start with the system-centred kiosk. Why does the order matter? What would happen to the comparison if everybody played the system-centred kiosk first?

## 2. Doors and burners: Signifiers and mappings in numbers &#91;[26](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/lecture.html#ref-26)&#93;

*Lecture: [Affordances, signifiers, mapping and feedback](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/lecture.html#affordances-signifiers-mapping-and-feedback)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "forms" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-01/"
               "NB01_interaction_loop_and_usability.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Every door and every burner records a choice and a time. Grouping a few hundred trials by condition, separating wrong first choices from slow correct ones and drawing the distributions is a short program. The same function serves both experiments, and it would serve any other experiment of this kind.

Parts C and D of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/lab.html) end with the button *Copy the results for Colab*. Paste the results of the doors into `MY_DOORS` and those of the cooktop into `MY_COOKTOP`. A variable left empty makes the code use a simulated class of twenty students, whose error rates and times are assumptions of this Colab notebook and not measurements.

In [ ]:
# Paste the results of lab Parts C and D between the triple quotes.
MY_DOORS = """
"""
MY_COOKTOP = """
"""

def simulated_doors(n_students=20, seed=RANDOM_STATE):
    """Three doors of each kind per student, with assumed error rates and times."""
    rng = np.random.default_rng(seed)
    # mean time of a correct first try in s, probability of a wrong first try
    setup = {"plate_push": (0.95, 0.03), "handle_pull": (1.00, 0.04),
             "handle_push": (1.35, 0.45), "handle_push_sign": (1.55, 0.12)}
    rows = []
    for student in range(1, n_students + 1):
        speed = rng.lognormal(0, 0.15)
        for door, kind in enumerate(rng.permutation(list(setup) * 3), start=1):
            mean_s, p_wrong = setup[kind]
            correct = rng.random() > p_wrong
            rt_ms = 1000 * speed * mean_s * rng.lognormal(0, 0.25)
            open_ms = rt_ms + (0 if correct else 1000 * rng.uniform(0.6, 1.4))
            need = "pull" if kind == "handle_pull" else "push"
            first = need if correct else {"push": "pull", "pull": "push"}[need]
            rows.append((student, door, kind, first, int(correct),
                         round(rt_ms), round(open_ms)))
    cols = ["student", "door", "kind", "first", "correct", "rt_ms", "open_ms"]
    return pd.DataFrame(rows, columns=cols)

def simulated_cooktop(n_students=20, seed=RANDOM_STATE):
    """Twelve trials per layout and student, with assumed means and error rates."""
    rng = np.random.default_rng(seed)
    setup = {"offset": (0.70, 0.02),          # mean time in s, error probability
             "square_row": (1.05, 0.15),
             "square_block": (0.78, 0.04)}
    square = ["BL", "BR", "FL", "FR"]
    names = {"offset": ["B1", "B2", "B3", "B4"], "square_row": square,
             "square_block": square}
    rows = []
    for student in range(1, n_students + 1):
        speed = rng.normal(1.0, 0.12)
        for layout, (mean_s, p_err) in setup.items():
            for trial, target in enumerate(rng.permutation(names[layout] * 3), 1):
                correct = rng.random() > p_err
                rt_ms = 1000 * speed * mean_s * rng.lognormal(0, 0.25)
                others = [b for b in names[layout] if b != target]
                choice = target if correct else rng.choice(others)
                rows.append((student, layout, trial, target, choice,
                             round(rt_ms), int(correct)))
    cols = ["student", "layout", "trial", "target", "choice", "rt_ms", "correct"]
    return pd.DataFrame(rows, columns=cols)

read = lambda text: pd.read_csv(io.StringIO(text.strip()))
doors = read(MY_DOORS) if MY_DOORS.strip() else simulated_doors()
cook = read(MY_COOKTOP) if MY_COOKTOP.strip() else simulated_cooktop()
print("Doors:", len(doors), "trials |", "cooktop:", len(cook), "trials")
doors.head()

**Python note.** `rng.permutation(list(setup) * 3)` repeats the four kinds of doors three times and shuffles them, so every student meets them in a different order. `{"push": "pull", "pull": "push"}[need]` looks up the opposite action in a small dictionary. A `lambda` is a short function without a name; `read` turns pasted text into a table.

In [ ]:
def condition_summary(trials, by):
    """Error rate of the first choices and mean reaction time of the correct ones."""
    correct = trials[trials["correct"] == 1]
    out = pd.DataFrame({
        "trials": trials.groupby(by).size(),
        "errors": trials.groupby(by)["correct"].apply(lambda c: int((c == 0).sum())),
        "mean_rt_s": correct.groupby(by)["rt_ms"].mean() / 1000})
    out["error_rate"] = out["errors"] / out["trials"]
    return out

def costs(summary, reference):
    """Cost of each condition against the reference: delta t and delta p."""
    if reference not in summary.index:
        return
    for name in summary.index.drop(reference):
        dt = summary.loc[name, "mean_rt_s"] - summary.loc[reference, "mean_rt_s"]
        dp = summary.loc[name, "error_rate"] - summary.loc[reference, "error_rate"]
        print(f"  {name:17s} delta t = {dt:+.2f} s, delta p = {dp:+.3f}")

door_sum = condition_summary(doors, "kind")
door_sum["mean_open_s"] = doors.groupby("kind")["open_ms"].mean() / 1000
cook_sum = condition_summary(cook, "layout")
print(door_sum.round(3), "\n")
print("cost of each door against the push plate:")
costs(door_sum, "plate_push")
print("\n", cook_sum.round(3), "\n")
print("cost of each layout against the offset burners:")
costs(cook_sum, "offset")

**Python note.** One function, `condition_summary`, serves both experiments, because both tables have a column for the condition, a column `correct` and a column `rt_ms`. `groupby(by)` splits the rows into one group per condition, and `size()`, `mean()` or `apply(...)` compute one value per group. `summary.index.drop` removes the reference from the list of conditions, and `:+.2f` prints a number with its sign.

In [ ]:
kinds = [k for k in ["plate_push", "handle_pull", "handle_push", "handle_push_sign"]
         if k in door_sum.index]
layouts = [l for l in ["offset", "square_row", "square_block"] if l in cook_sum.index]
fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.8))
ax[0].bar(range(len(kinds)), door_sum.loc[kinds, "error_rate"], color="#5A4BB5")
ax[0].set_xticks(range(len(kinds)), [k.replace("_", "\n") for k in kinds])
ax[0].set_ylabel("error rate of the first try")
right = ax[0].twinx()
right.plot(range(len(kinds)), door_sum.loc[kinds, "mean_open_s"], "o-", color="#C98510")
right.set_ylabel("mean time to open (s)", color="#C98510")
ax[0].set_title("Doors: bars errors, line time to open")
correct = cook[cook["correct"] == 1]
ax[1].boxplot([correct.loc[correct["layout"] == l, "rt_ms"] / 1000 for l in layouts],
              showfliers=False)
ax[1].set_xticks(range(1, len(layouts) + 1), layouts)
ax[1].set_ylabel("reaction time of correct trials (s)")
ax[1].set_title("Cooktop: reaction times by layout")
plt.tight_layout()
plt.show()

**Python note.** `ax[0].twinx()` adds a second vertical axis on the right of the same panel, so that error rates and times share one horizontal axis. `"o-"` draws points joined by lines, and `k.replace("_", "\n")` breaks the long names of the doors into lines.

**Exercise C.** Store the error rate of the first tries at the doors of the kind `handle_push` in `p_err_door`, as a number between 0 and 1. These doors have a pull handle but open by pushing. With your own trials, walk through the whole corridor first.

In [ ]:
p_err_door = None   # your answer

check("Exercise C", p_err_door)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_err_door = door_sum.loc["handle_push", "error_rate"]

check("Exercise C", p_err_door)

**Observe and reflect.** The Push sign lowers the errors at the door with the conflicting handle, but does it remove them, and what does it cost in time? The row of knobs of the cooktop carries small icons that show the burner of each knob. Why do labels and icons help less than hardware and layouts that need no explanation?

## 3. A flow as a chain of steps

*Lecture: [A task as a chain of steps](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/lecture.html#a-task-as-a-chain-of-steps)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "door_sum" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-01/"
               "NB01_interaction_loop_and_usability.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Probabilities of several steps multiply, and a small error per step becomes a large one over a flow. Python computes the three formulas for any flow and checks them with a simulation of 10,000 patients, which no calculation by hand can do.

**From the formulas to the code.** A flow is a list of dictionaries, one per step, with the keys `"t"`, `"e"`, `"r"` and `"a"` for $t_k$, $e_k$, $r_k$ and $a_k$. The product $\prod$ becomes a loop with `p *= ...`, and the sum $\sum$ becomes `sum(...)` over the steps.

In [ ]:
def p_smooth(flow):
    """P_0: probability that no step of the flow has a slip."""
    p = 1.0
    for step in flow:
        p *= 1 - step["e"]
    return p

def expected_time(flow):
    """E[T]: expected duration in seconds when every slip is repaired."""
    return sum(step["t"] + step["e"] * step["r"] for step in flow)

def p_complete(flow):
    """P_c: probability of completing the flow; people give up only after a slip."""
    p = 1.0
    for step in flow:
        p *= 1 - step["e"] * step["a"]
    return p

EXAMPLE = [
    {"step": "Choose the department", "t": 12, "e": 0.20, "r": 10, "a": 0.20},
    {"step": "Pick a date",           "t": 10, "e": 0.15, "r": 12, "a": 0.20},
    {"step": "Confirm",               "t": 4,  "e": 0.12, "r": 25, "a": 0.20},
]
print(f"P_0  = {p_smooth(EXAMPLE):.3f}")
print(f"E[T] = {expected_time(EXAMPLE):.1f} s")
print(f"P_c  = {p_complete(EXAMPLE):.3f}")

**Python note.** `p *= x` is short for `p = p * x`. `sum(... for step in flow)` adds one value per step; the expression inside is a generator, a loop written in one line.

In [ ]:
def simulate(flow, n_patients=10_000, seed=RANDOM_STATE):
    """Sends simulated patients through the flow. Each step slips with probability
    e; after a slip the patient gives up with probability a, otherwise the slip is
    repaired in r seconds."""
    rng = np.random.default_rng(seed)
    time_s = np.zeros(n_patients)
    active = np.ones(n_patients, dtype=bool)
    slips = np.zeros(n_patients, dtype=int)
    gave_up_at = np.full(n_patients, -1)
    for k, step in enumerate(flow, start=1):
        slip = active & (rng.random(n_patients) < step["e"])
        quits = slip & (rng.random(n_patients) < step["a"])
        time_s[active] += step["t"]
        time_s[slip & ~quits] += step["r"]
        slips += slip
        gave_up_at[quits] = k
        active &= ~quits
    return pd.DataFrame({"time_s": time_s, "slips": slips, "completed": active,
                         "gave_up_at": gave_up_at})

sim = simulate(EXAMPLE)
done = sim[sim["completed"]]
compare = pd.DataFrame({
    "formula": [p_smooth(EXAMPLE), p_complete(EXAMPLE), expected_time(EXAMPLE)],
    "simulation": [(sim["slips"] == 0).mean(), sim["completed"].mean(),
                   done["time_s"].mean()]},
    index=["no slip at all, P_0", "completed, P_c", "mean time in s, E[T]"])
compare.round(3)

**Python note.** The simulation handles all patients at once with NumPy arrays &#91;[6](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/lecture.html#ref-6)&#93;. A small table, `compare`, puts the formulas and the simulation side by side. `rng.random(n_patients) < e` is an array of `True` and `False`, one per patient, and `&`, `|` and `~` combine such arrays element by element as and, or and not. `time_s[active] += step["t"]` adds the time only for the patients who are still in the flow.

In [ ]:
n = np.arange(1, len(sim) + 1)
running_share = sim["completed"].cumsum() / n
plt.figure(figsize=(7, 3.2))
plt.semilogx(n, running_share, color="#5A4BB5", label="simulated share completed")
plt.axhline(p_complete(EXAMPLE), color="#C98510", ls="--", label="formula P_c")
plt.xlabel("number of simulated patients (log scale)")
plt.ylabel("share completed")
plt.ylim(0.8, 1.0)
plt.legend()
plt.tight_layout()
plt.show()

**Python note.** `cumsum()` adds the values one by one, so `cumsum() / n` is the share after 1, 2, 3 and more patients. `plt.semilogx` uses a logarithmic horizontal axis, on which 10, 100 and 1000 are equally spaced.

In [ ]:
# P_0 = (1 - e)^K as a surface over the slip probability e and the number of steps K
e_grid = np.linspace(0, 0.3, 31)
K_grid = np.arange(1, 16)
E_grid, K_mesh = np.meshgrid(e_grid, K_grid)
P0_surface = (1 - E_grid) ** K_mesh
fig = plt.figure(figsize=(7, 4.8))
ax = fig.add_subplot(projection="3d")
ax.plot_surface(E_grid, K_mesh, P0_surface, cmap="viridis", edgecolor="none",
                alpha=0.95)
ax.set_xlabel("slip probability e")
ax.set_ylabel("number of steps K")
ax.set_zlabel("P_0")
ax.view_init(elev=28, azim=35)
ax.set_title("The chance of a task without any slip")
plt.tight_layout()
plt.show()

**Python note.** `np.meshgrid` turns two lists of values into two grids, so that every combination of $e$ and $K$ has one point. `**` raises to a power element by element. `projection="3d"` makes a three-dimensional axis, `plot_surface` draws the grid as a surface, and `view_init` sets the angle of view. In Colab the figure is static; the interactive lab turns 3D scenes freely.

**Exercise D.** Half of all slips now end with giving up. Make a copy of `EXAMPLE` with $a_k = 0.5$ for every step and store its $P_c$ in `pc_half`. The list comprehension of the first line already makes the copy and keeps `EXAMPLE` unchanged.

In [ ]:
half = [dict(step, a=0.5) for step in EXAMPLE]   # a copy with a = 0.5
pc_half = None   # your answer

check("Exercise D", pc_half)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
half = [dict(step, a=0.5) for step in EXAMPLE]
pc_half = p_complete(half)

check("Exercise D", pc_half)

**Observe and reflect.** The simulated mean time of the completed patients is a little below the formula for $\mathrm{E}[T]$. Which patients does the formula count that the simulation leaves out? Why does the gap grow when $a_k$ grows?

## 4. Which step costs the most?

*Lecture: [A task as a chain of steps](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/lecture.html#a-task-as-a-chain-of-steps)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "simulate" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-01/"
               "NB01_interaction_loop_and_usability.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Eight what-if questions, one per step, each need two formulas. A loop answers all of them at once and ranks the steps, which tells the designer where to start.

The booking flow of the clinic app, `CLINIC_FLOW`, is defined in Section 0. The next cell halves the slip probability of one step at a time and records how much $P_c$ and $\mathrm{E}[T]$ improve.

In [ ]:
base_pc, base_et = p_complete(CLINIC_FLOW), expected_time(CLINIC_FLOW)
rows = []
for k, step in enumerate(CLINIC_FLOW, start=1):
    better = [dict(s) for s in CLINIC_FLOW]       # a copy of the flow
    better[k - 1]["e"] = step["e"] / 2            # halve the slip probability
    rows.append({"step": k, "action": step["step"],
                 "gain_pc_points": 100 * (p_complete(better) - base_pc),
                 "time_saved_s": base_et - expected_time(better)})
gains = pd.DataFrame(rows).set_index("step").round(2)
gains.sort_values("gain_pc_points", ascending=False)

**Python note.** `[dict(s) for s in CLINIC_FLOW]` copies every dictionary, so that changing the copy leaves the original flow intact. `set_index("step")` uses the step number as the row label, and `sort_values(..., ascending=False)` puts the largest gain first.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].barh(gains.index.astype(str), gains["gain_pc_points"], color="#5A4BB5")
ax[0].set_xlabel("gain in P_c (percentage points)")
ax[0].set_ylabel("step")
ax[0].invert_yaxis()
ax[1].barh(gains.index.astype(str), gains["time_saved_s"], color="#C98510")
ax[1].set_xlabel("time saved in E[T] (s)")
ax[1].invert_yaxis()
fig.suptitle("Halving the slip probability of one step at a time")
plt.tight_layout()
plt.show()

**Python note.** `plt.subplots(1, 2)` creates a figure with two panels side by side, `ax[0]` and `ax[1]`. `barh` draws horizontal bars, and `invert_yaxis()` puts step 1 at the top.

**Explore.** Move the sliders and watch the three values change against the baseline. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as W

def explore(step=2, e=0.25, a=0.10, r=15):
    flow_x = [dict(s) for s in CLINIC_FLOW]
    flow_x[step - 1].update(e=e, a=a, r=r)
    print(f"Step {step}: {flow_x[step - 1]['step']}")
    values = [("P_0 ", p_smooth), ("E[T]", expected_time), ("P_c ", p_complete)]
    for name, f in values:
        print(f"{name} = {f(flow_x):7.3f}   baseline {f(CLINIC_FLOW):7.3f}")

W.interact(explore,
           step=W.IntSlider(value=2, min=1, max=8, description="step"),
           e=W.FloatSlider(value=0.25, min=0.0, max=0.5, step=0.01, description="e"),
           a=W.FloatSlider(value=0.10, min=0.0, max=1.0, step=0.05, description="a"),
           r=W.IntSlider(value=15, min=0, max=60, step=1, description="r (s)"));

**Python note.** `W.interact(explore, ...)` draws one slider for each argument of `explore` and calls the function again whenever a slider moves. The sliders are created with keyword arguments such as `value=` and `min=`. `update(e=e, a=a, r=r)` changes several entries of a dictionary at once. The list of pairs in the loop holds a label and a function, so one loop prints all three values.

**Observe and reflect.** Halving the slip probability helps step 2 most. Set step 2 to $e = 0.05$ with the sliders, which is what a visible Appointments tab could achieve. How much of the gap to $P_c = 1$ closes, and which step becomes the weakest next?

## 5. The clinic app, stage 1

*Lecture: [The clinic app, stage 1: A baseline of the booking flow](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/lecture.html#the-clinic-app-stage-1)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "gains" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-01/"
               "NB01_interaction_loop_and_usability.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The same functions give the baseline of the clinic app. Every later week tries to improve these numbers, and Week 9 measures the new design against them.

In [ ]:
BASELINE = {
    "P_0": p_smooth(CLINIC_FLOW),
    "E[T]_s": expected_time(CLINIC_FLOW),
    "P_c": p_complete(CLINIC_FLOW),
}
sim_clinic = simulate(CLINIC_FLOW)
BASELINE["P_c_simulated"] = sim_clinic["completed"].mean()
completed = sim_clinic["completed"]
BASELINE["E[T]_simulated_s"] = sim_clinic.loc[completed, "time_s"].mean()
table = pd.DataFrame(CLINIC_FLOW)
table.index = range(1, len(table) + 1)
table["e*a (gives up)"] = table["e"] * table["a"]
table["e*r (repair s)"] = table["e"] * table["r"]
print(f"P_0 = {BASELINE['P_0']:.3f} | E[T] = {BASELINE['E[T]_s']:.1f} s "
      f"| sum of t = {table['t'].sum()} s | P_c = {BASELINE['P_c']:.3f}")
print(f"10,000 simulated patients: {BASELINE['P_c_simulated']:.3f} completed, "
      f"mean time of the completed {BASELINE['E[T]_simulated_s']:.1f} s")
table

**Python note.** `pd.DataFrame(CLINIC_FLOW)` turns the list of dictionaries into a table with one column per key. `table.index = range(1, 9)` numbers the rows from 1, like the steps of the lecture. A new column is created by assigning to a new name, and arithmetic on columns works row by row.

In [ ]:
remaining = 100 * np.cumprod([1 - s["e"] * s["a"] for s in CLINIC_FLOW])
steps = np.arange(1, len(CLINIC_FLOW) + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(steps, remaining, color="#5A4BB5")
ax[0].set_ylim(85, 100.5)
ax[0].set_xlabel("step")
ax[0].set_ylabel("patients still in the flow (%)")
ax[0].set_title(f"Expected share after each step; P_c = {BASELINE['P_c']:.3f}")
ax[1].bar(steps, table["e*r (repair s)"], color="#C98510")
ax[1].set_xlabel("step")
ax[1].set_ylabel("expected repair time (s)")
ax[1].set_title("Expected repair time e_k r_k of each step")
plt.tight_layout()
plt.show()

**Python note.** `np.cumprod` multiplies the values one by one and keeps every partial product. Each partial product is the share of patients still in the flow after step 1, 2, 3 and so on.

**Exercise E.** Store the number of the step with the largest probability of giving up, $e_k a_k$, in `worst_step`, as a whole number from 1 to 8.

In [ ]:
worst_step = None   # your answer

check("Exercise E", worst_step)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
worst_step = int(table["e*a (gives up)"].idxmax())

check("Exercise E", worst_step)

**Observe and reflect.** The baseline loses about eleven patients in a hundred. Which of the lost patients would a hospital never notice in its own data, and how could the app record them? Week 9 returns to this question.

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "BASELINE" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-01/"
               "NB01_interaction_loop_and_usability.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Changing one name in the code moves the whole analysis to another app. This is why the model was written as functions of a flow rather than as numbers typed into a calculator.

Each flow below has illustrative values, as in the lecture. Set `APP` to one of the five names and run the two cells.

In [ ]:
APP = "banking"
# choose: clinic, banking, e_government, food_delivery, course_registration

def make_flow(rows):
    return [dict(step=s, t=t, e=e, r=r, a=a) for s, t, e, r, a in rows]

FLOWS = {
    "clinic": CLINIC_FLOW,
    "banking": make_flow([
        ("Log in with a one-time code", 25, 0.08, 30, 0.15),
        ("Open Payments", 5, 0.10, 8, 0.05),
        ("Choose the company from a list", 10, 0.12, 12, 0.10),
        ("Type the subscriber number", 18, 0.15, 20, 0.10),
        ("Check the amount", 6, 0.05, 10, 0.05),
        ("Confirm with a password", 8, 0.06, 15, 0.20)]),
    "e_government": make_flow([
        ("Log in through the national portal", 30, 0.12, 40, 0.25),
        ("Search for the service", 15, 0.30, 20, 0.15),
        ("Read the instructions", 40, 0.10, 30, 0.10),
        ("Fill in the form", 60, 0.20, 45, 0.15),
        ("Download the document", 8, 0.10, 15, 0.10)]),
    "food_delivery": make_flow([
        ("Choose a restaurant", 30, 0.05, 10, 0.02),
        ("Add dishes to the basket", 25, 0.10, 10, 0.05),
        ("Enter the address", 20, 0.12, 25, 0.10),
        ("Pay with a card", 25, 0.10, 30, 0.20)]),
    "course_registration": make_flow([
        ("Log in to the student system", 20, 0.10, 30, 0.10),
        ("Open the course list", 10, 0.20, 15, 0.05),
        ("Find the course by its code", 25, 0.15, 20, 0.10),
        ("Check for a timetable clash", 20, 0.25, 40, 0.15),
        ("Add the course", 5, 0.08, 10, 0.05),
        ("Submit for the approval of the advisor", 8, 0.10, 20, 0.10)]),
}
app_flow = FLOWS[APP]
print(f"{APP}: {len(app_flow)} steps")
for i, st in enumerate(app_flow, start=1):
    print(f"  {i}. {st['step']:<40} t={st['t']:>3} s  e={st['e']:.2f}  "
          f"r={st['r']:>3} s  a={st['a']:.2f}")

**Python note.** `make_flow` turns rows of five values into the list of dictionaries that the functions of Section 3 expect. `FLOWS[APP]` selects one flow from the dictionary of flows by its name.

In [ ]:
loss = np.array([st["e"] * st["a"] for st in app_flow])
worst = int(np.argmax(loss)) + 1
print(f"P_0 = {p_smooth(app_flow):.3f} | E[T] = {expected_time(app_flow):.1f} s "
      f"| P_c = {p_complete(app_flow):.3f}")
print(f"weakest step: {worst}, {app_flow[worst - 1]['step']} "
      f"(e*a = {loss[worst - 1]:.3f})")
plt.figure(figsize=(7, 3.2))
plt.bar(range(1, len(app_flow) + 1), 100 * np.cumprod(1 - loss), color="#5A4BB5")
plt.ylim(min(100 * np.cumprod(1 - loss)) - 3, 100.5)
plt.xlabel("step")
plt.ylabel("patients or users still in the flow (%)")
plt.title(f"The funnel of the flow: {APP}")
plt.tight_layout()
plt.show()

**Python note.** `np.argmax` returns the position of the largest value, counted from 0, so 1 is added to obtain the step number. `1 - loss` subtracts every element of the array from 1 at once.

**Observe and reflect.** Compare the weakest steps of two flows. Is the weakest step a gulf of execution or a gulf of evaluation? Which change of the interface clinic would help it?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Write unit tests for `p_smooth`, `expected_time` and `p_complete`, including a flow without steps and a step with $e = 1$, and correct any function that fails. |
| Web development | Model the checkout of an online shop with six steps, then compute how a guest checkout without an account changes $P_c$ and $\mathrm{E}[T]$. |
| Mobile development | Model the five onboarding screens of a mobile app. Which screen would you remove first, and what does the model predict? |
| Game development | Model the tutorial of a game as a chain of eight steps. How many players reach the first level if every step loses 3 percent of them? |
| Embedded systems and the Internet of Things | A smart thermostat has two buttons and a small display. List the steps of setting a weekly schedule and the gulf behind each likely slip. |
| Data science | Draw 1000 bootstrap samples of the rounds of each kiosk in Section 1, with replacement, and report the spread of the difference in the mean time of the completed rounds. |
| Cybersecurity | Two-factor login adds a step with $t = 15$ s, $e = 0.08$, $r = 20$ s and $a = 0.10$. Compute its cost for the clinic app and propose a design that keeps the security at a smaller cost. |
| Health informatics | The nurse at the front desk books for others all day. Lower the times and slip probabilities of the clinic flow for an expert user and compare the bottlenecks with those of the patients. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-01).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>